# Método no congruencial: generación de números pseudoaleatorios por cuadrados medios

**Materia:** Optimización estocástica / Simulación  
**Método:** Cuadrados medios (no congruencial)  
**Objetivo:** Comprender e implementar un generador de números pseudoaleatorios, observar el cálculo paso a paso, producir valores normalizados y estudiar sus limitaciones.

Este notebook se basa en el método de **cuadrados medios** presentado en la sesión *5.1. Generación de Números PseudoAleatorios*. En esa presentación también aparece el método del producto medio, pero aquí desarrollaremos **un solo método no congruencial**, como se solicita.

**Al terminar podrás:**
1. Calcular a mano una iteración.
2. Generar tantos valores como indiques, sin depender de un generador aleatorio incorporado.
3. Reanudar la secuencia sin perder el estado anterior.
4. Identificar repeticiones y ciclos (incluida la degeneración).
5. Examinar histogramas y correlación serial de forma exploratoria.

> **Nota:** los números obtenidos son *pseudoaleatorios* y el método de cuadrados medios es principalmente didáctico. No es adecuado para criptografía ni para simulaciones que exijan grandes cantidades de números de alta calidad.

## 1. ¿Qué es un método no congruencial?

Un **generador pseudoaleatorio** comienza con una **semilla** y aplica una regla determinista para construir una secuencia que intenta comportarse como aleatoria. Si volvemos a utilizar la misma semilla, obtenemos exactamente la misma secuencia.

Los métodos **no congruenciales**, como cuadrados medios o producto medio, no utilizan una relación de recurrencia del tipo **generador congruencial lineal**:

$$X_{i+1}=(aX_i+c)\bmod m.$$

En **cuadrados medios**, el siguiente estado se obtiene elevando el anterior al cuadrado y extrayendo sus dígitos centrales.

La presentación identifica como propiedades deseables la **uniformidad**, la **no correlación serial**, un **período largo** y la **reproducibilidad**. Estas características se deben comprobar; no quedan garantizadas simplemente por utilizar este método.

## 2. Regla del método de cuadrados medios

Para trabajar con una cantidad **par** de dígitos, aquí elegimos **d = 4**:

1. Escoge una semilla entera de cuatro dígitos (o un entero más pequeño, completándolo con ceros por la izquierda cuando corresponda).
2. Eleva el estado actual al cuadrado: **cuadrado = Xᵢ²**.
3. Representa el resultado con **2d = 8 dígitos**, agregando ceros a la izquierda si hacen falta.
4. Extrae los **d = 4 dígitos centrales**; su valor será **Xᵢ₊₁**.
5. Normaliza: **Uᵢ₊₁ = Xᵢ₊₁ / 10ᵈ**. Para d = 4, divide entre 10 000.
6. Repite el procedimiento usando el nuevo estado.

De este modo se obtienen valores **0 ≤ Uᵢ < 1**. Que los valores estén en ese intervalo no significa que tengan una distribución uniforme o sean independientes.

Para que la extracción de los dígitos centrales sea inequívoca, en esta implementación usamos solamente un número **par de dígitos**.

## 3. Ejemplo resuelto a mano: semilla 3500

Consideremos **X₀ = 3500** y **d = 4**.

**Primera iteración:**
- Cuadrado: **3500² = 12 250 000**.
- Ocho dígitos: **12250000**.
- Eliminamos dos dígitos a cada extremo: **12 | 2500 | 00**.
- Nuevo estado: **X₁ = 2500**.
- Número normalizado: **U₁ = 2500 / 10000 = 0.25**.

**Segunda iteración:**
- Cuadrado: **2500² = 6 250 000**.
- Lo completamos a ocho dígitos: **06250000**.
- Dígitos centrales: **06 | 2500 | 00**.
- Nuevo estado: **X₂ = 2500** y **U₂ = 0.25**.

¡El resultado ya no cambia! Este ejemplo de la presentación ilustra la **degeneración a un punto fijo**.

Vamos a reproducir ambas iteraciones en Python, sin utilizar ninguna biblioteca de aleatoriedad.

In [ ]:
semilla_manual = 3500
digitos_manual = 4
ancho = 2 * digitos_manual
inicio = digitos_manual // 2

for paso in range(1, 3):
    cuadrado = semilla_manual ** 2
    cadena = str(cuadrado).zfill(ancho)
    centrales = cadena[inicio:inicio + digitos_manual]
    siguiente = int(centrales)
    uniforme = siguiente / (10 ** digitos_manual)
    print(
        f"Paso {paso}: {semilla_manual}² = {cuadrado} "
        f"→ {cadena[:inicio]}|{centrales}|{cadena[inicio + digitos_manual:]} "
        f"→ X={siguiente:0{digitos_manual}d}, U={uniforme:.4f}"
    )
    semilla_manual = siguiente

## 4. Importar herramientas de análisis

Utilizamos:
- **pandas**: mostrar las iteraciones en una tabla.
- **numpy**: resumir algunos resultados numéricos.
- **matplotlib**: construir el histograma y el diagrama de pares sucesivos.

No emplearemos `random` ni `numpy.random` para **generar** la secuencia: todos los valores serán calculados por cuadrados medios.

Si hace falta instalar dependencias en tu entorno, ejecuta en una terminal: `pip install numpy pandas matplotlib`.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

## 5. Implementación completa del generador

La clase `CuadradosMedios` conserva:
- `estado`: el último número producido.
- `paso`: el número total de iteraciones realizadas.
- `posiciones`: la primera iteración en que apareció cada estado.
- `ciclo`: información del primer ciclo detectado.

El método `siguiente()` construye **una fila detallada** del cálculo. El método `generar(cantidad)` repite esa operación y devuelve una tabla.

**Importante:** establecer `detener_en_ciclo=True` detiene el lote al detectar una repetición. Con `False` puedes seguir generando (aunque los valores repetidos ya no aporten nueva información).

In [ ]:
class CuadradosMedios:
    """Generador pseudoaleatorio NO congruencial por el método de cuadrados medios."""

    def __init__(self, semilla: int, digitos: int = 4):
        if not isinstance(digitos, int) or digitos < 2 or digitos % 2 != 0:
            raise ValueError("digitos debe ser un entero par mayor o igual que 2.")
        if not isinstance(semilla, int) or not (0 <= semilla < 10 ** digitos):
            raise ValueError(f"semilla debe ser un entero entre 0 y {10 ** digitos - 1}.")

        self.semilla = semilla
        self.digitos = digitos
        self.estado = semilla
        self.paso = 0
        self.posiciones = {semilla: 0}
        self.ciclo = None

    def siguiente(self) -> dict:
        # Paso 1: leer el estado actual.
        anterior = self.estado

        # Paso 2: calcular el cuadrado sin utilizar random().
        cuadrado = anterior ** 2

        # Paso 3: completar con ceros a la izquierda hasta tener 2*d dígitos.
        cadena_cuadrado = str(cuadrado).zfill(2 * self.digitos)

        # Paso 4: obtener los d dígitos centrales.
        inicio = self.digitos // 2
        texto_central = cadena_cuadrado[inicio:inicio + self.digitos]
        nuevo = int(texto_central)

        # Paso 5: convertir el estado a un número en [0, 1).
        u = nuevo / (10 ** self.digitos)

        # Paso 6: registrar repeticiones para identificar ciclos.
        self.paso += 1
        repetido = nuevo in self.posiciones
        inicio_ciclo = self.posiciones[nuevo] if repetido else None
        periodo = self.paso - inicio_ciclo if repetido else None

        if repetido:
            if self.ciclo is None:
                self.ciclo = (inicio_ciclo, periodo)
        else:
            self.posiciones[nuevo] = self.paso

        # Paso 7: conservar el nuevo estado para continuar después.
        self.estado = nuevo

        return {
            "Paso": self.paso,
            "X_anterior": anterior,
            "Cuadrado": cuadrado,
            "Cuadrado_2d": cadena_cuadrado,
            "Centrales": texto_central,
            "X_nuevo": nuevo,
            "U": u,
            "Repetido": repetido,
            "Inicio_ciclo": inicio_ciclo,
            "Periodo": periodo,
        }

    def generar(self, cantidad: int, detener_en_ciclo: bool = False) -> pd.DataFrame:
        """Produce un lote y permite continuar desde el último estado en otra llamada."""
        if not isinstance(cantidad, int) or cantidad < 1:
            raise ValueError("cantidad debe ser un entero positivo.")

        filas = []
        for _ in range(cantidad):
            fila = self.siguiente()
            filas.append(fila)
            if detener_en_ciclo and fila["Repetido"]:
                break

        return pd.DataFrame(filas)

## 6. Elegir los parámetros y generar un primer lote

Puedes modificar solamente estas variables para experimentar:

- **SEMILLA:** número inicial, entre 0 y 9999 si d = 4.
- **DIGITOS:** cantidad par de cifras con las que trabaja el algoritmo.
- **PRIMER_LOTE:** cuántos números se generan al principio.
- **SEGUNDO_LOTE:** cuántos números adicionales se solicitan después.

La tabla deja visible todo el proceso: cuadrado, representación con ceros, dígitos centrales y valor normalizado.

Usaremos `5735` como semilla de prueba. La clase no depende de un valor particular: puedes cambiarla por cualquier semilla válida.

In [ ]:
SEMILLA = 5735
DIGITOS = 4
PRIMER_LOTE = 12
SEGUNDO_LOTE = 8

generador = CuadradosMedios(semilla=SEMILLA, digitos=DIGITOS)
primer_lote = generador.generar(PRIMER_LOTE)

print(f"Semilla: {SEMILLA} | Dígitos: {DIGITOS}")
print(f"Cantidad en el primer lote: {len(primer_lote)}")
display(primer_lote)

## 7. Continuar la misma secuencia sin reiniciar

Una propiedad importante para la simulación es **conservar el estado** del generador. No debemos volver a crearlo con la misma semilla cuando queremos números *nuevos* de la misma secuencia.

A continuación llamamos al **mismo objeto** `generador` para generar otro lote. Después unimos ambas tablas. Observa que la columna `Paso` continúa donde terminó el primer lote, y que el primer estado usado en el segundo lote es el último valor del primero.

In [ ]:
segundo_lote = generador.generar(SEGUNDO_LOTE)
datos = pd.concat([primer_lote, segundo_lote], ignore_index=True)

print(f"Pasos totales generados: {generador.paso}")
print(f"Último estado almacenado: {generador.estado:0{DIGITOS}d}")
print(f"Primer valor del segundo lote: U={segundo_lote.iloc[0]['U']:.4f}")
display(segundo_lote)
print("Secuencia normalizada completa:")
print(datos["U"].round(DIGITOS).tolist())

## 8. Visualización: histograma y correlación serial

Una buena secuencia pseudoaleatoria debería distribuirse razonablemente en el intervalo [0, 1) y no mostrar dependencias sistemáticas entre números sucesivos.

El **histograma** nos ayuda a observar frecuencias. El **diagrama de dispersión de (Uᵢ, Uᵢ₊₁)** permite buscar patrones visibles entre observaciones consecutivas.

Ambos análisis son **exploratorios**, no una prueba de que los valores sean realmente uniformes o independientes. Usar una muestra pequeña limita mucho las conclusiones.

In [ ]:
u = datos["U"].to_numpy(dtype=float)

fig, ejes = plt.subplots(1, 2, figsize=(13, 4))
ejes[0].hist(u, bins=10, range=(0, 1), edgecolor="black")
ejes[0].set_title("Histograma de U")
ejes[0].set_xlabel("Número pseudoaleatorio U")
ejes[0].set_ylabel("Frecuencia")
ejes[0].set_xlim(0, 1)

ejes[1].scatter(u[:-1], u[1:], alpha=0.8)
ejes[1].set_title("Pares sucesivos")
ejes[1].set_xlabel("U_i")
ejes[1].set_ylabel("U_(i+1)")
ejes[1].set_xlim(0, 1)
ejes[1].set_ylim(0, 1)

plt.tight_layout()
plt.show()

## 9. Estadísticas descriptivas de la serie

Como referencia, una variable verdaderamente uniforme en [0,1) tiene:
- **Media teórica:** 0.5.
- **Varianza teórica:** 1/12 ≈ 0.08333.
- **Correlación entre observaciones independientes:** 0.

Calcularemos la media, la varianza poblacional y la correlación de pares sucesivos. Las diferencias **no significan por sí solas** que la secuencia haya aprobado o reprobado una prueba estadística. Para una validación formal se requieren procedimientos y tamaños muestrales apropiados.

In [ ]:
media = float(np.mean(u))
varianza = float(np.var(u))
correlacion = float(pd.Series(u).autocorr(lag=1)) if len(u) > 2 else float("nan")

print(f"Tamaño de la muestra: {len(u)}")
print(f"Media muestral: {media:.6f}  |  Referencia uniforme: 0.500000")
print(f"Varianza (divisor n): {varianza:.6f}  |  Referencia uniforme: {1/12:.6f}")
print(f"Correlación serial (rezago 1): {correlacion:.6f}  |  Referencia independiente: 0")
print(f"Valores diferentes de U: {len(np.unique(u))} de {len(u)}")

if generador.ciclo is None:
    print("No se detectó un ciclo en los pasos generados hasta ahora.")
else:
    inicio_ciclo, periodo = generador.ciclo
    print(f"Se detectó un ciclo que empieza en el paso {inicio_ciclo} y tiene longitud {periodo}.")

## 10. Demostrar una limitación: el método puede degenerar

La presentación advierte que **cuadrados medios puede degenerar rápidamente**. En particular, con semilla 3500 y cuatro dígitos la secuencia cae en el estado 2500 y ya no sale:

**3500 → 2500 → 2500 → 2500 → ...**

El generador identifica la primera repetición y, si lo indicamos, deja de calcular nuevos valores. En este caso, el **período del ciclo es 1**.

Esto muestra por qué **no basta con ver números entre 0 y 1**: una secuencia puede normalizarse correctamente y aun así ser inadecuada para simular fenómenos aleatorios.

In [ ]:
generador_degenerado = CuadradosMedios(semilla=3500, digitos=4)
tabla_degenerada = generador_degenerado.generar(
    cantidad=20, detener_en_ciclo=True
)
display(tabla_degenerada)
print("Primer ciclo detectado (paso inicial, longitud):", generador_degenerado.ciclo)
print("Números normalizados:", tabla_degenerada["U"].tolist())

## 11. Pruebas automáticas de la implementación

Revisamos cuatro propiedades del código:
1. **Extracción central correcta:** 1234² = 01522756 → 5227.
2. **Normalización correcta:** 5227/10000 = 0.5227.
3. **Degeneración detectada:** 3500 → 2500 → 2500, período 1.
4. **Continuidad:** generar 3 valores y luego 4 debe dar lo mismo que generar 7 de una sola vez.

Estas pruebas comprueban el **funcionamiento del algoritmo**, no la calidad estadística general de sus secuencias.

In [ ]:
# Prueba 1 y 2: cuadrado, cifras centrales y normalización
prueba = CuadradosMedios(1234, 4)
fila = prueba.siguiente()
assert fila["Cuadrado_2d"] == "01522756"
assert fila["Centrales"] == "5227"
assert fila["X_nuevo"] == 5227
assert abs(fila["U"] - 0.5227) < 1e-12

# Prueba 3: detectar correctamente el ciclo de longitud 1
deg = CuadradosMedios(3500, 4)
serie_deg = deg.generar(10, detener_en_ciclo=True)
assert serie_deg["X_nuevo"].tolist() == [2500, 2500]
assert deg.ciclo == (1, 1)

# Prueba 4: dos lotes equivalen a una sola generación larga
por_lotes = CuadradosMedios(5735, 4)
parte_a = por_lotes.generar(3)
parte_b = por_lotes.generar(4)
junto = pd.concat([parte_a, parte_b], ignore_index=True)

de_una_vez = CuadradosMedios(5735, 4).generar(7)
assert junto["X_nuevo"].tolist() == de_una_vez["X_nuevo"].tolist()
assert np.allclose(junto["U"].to_numpy(), de_una_vez["U"].to_numpy())

# Pruebas de validación de parámetros incorrectos
for semilla_mala, digitos_malos in [(10000, 4), (100, 3)]:
    try:
        CuadradosMedios(semilla_mala, digitos_malos)
        raise AssertionError("No se rechazó una configuración inválida.")
    except ValueError:
        pass

print("Todas las pruebas del generador pasaron correctamente.")

## 12. Entrada opcional de parámetros mediante teclado

Si el ejercicio pide explícitamente que el usuario introduzca la semilla, los dígitos y la cantidad, ejecuta la función siguiente y luego **descomenta** la última línea.

Por defecto no se solicita información al abrir o ejecutar todo el notebook, para que pueda ejecutarse de forma automática sin quedarse esperando una entrada.

In [ ]:
def generar_desde_teclado():
    semilla = int(input("Semilla (ejemplo: 5735): "))
    digitos = int(input("Cantidad PAR de dígitos (ejemplo: 4): "))
    cantidad = int(input("Cantidad de números a generar: "))
    objeto = CuadradosMedios(semilla=semilla, digitos=digitos)
    tabla = objeto.generar(cantidad, detener_en_ciclo=True)
    display(tabla)
    print("Último estado:", objeto.estado, "| Primer ciclo:", objeto.ciclo)
    return objeto, tabla

# Para usarlo interactivamente, quita el # de la siguiente línea:
# mi_generador, mi_tabla = generar_desde_teclado()

## 13. Conclusiones

1. **Cuadrados medios es no congruencial:** calcula cada número mediante cuadrados y extracción de los dígitos centrales, no mediante una regla de congruencia lineal.
2. **Es reproducible:** una misma semilla y una misma cantidad de dígitos producen la misma secuencia.
3. **Puede continuar por lotes:** basta con preservar el objeto y su estado, sin reiniciar el generador.
4. **Tiene una limitación importante:** puede entrar pronto en ciclos cortos o puntos fijos; la semilla 3500 lo muestra con claridad.
5. **La normalización no garantiza calidad estadística:** revisar uniformidad, independencia y período es fundamental antes de usar cualquier generador en simulación.

**Referencia de clase:** *5.1. Generación de Números PseudoAleatorios* — sección «Generadores no congruenciales», «Método del cuadrado medio» y «Análisis».

**Actividad sugerida:** cambia `SEMILLA` por 1000, 2500 o 9876, vuelve a ejecutar desde el comienzo y compara cuánto tardan en aparecer las repeticiones.